In [2]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 18.0 MB/s eta 0:00:00


In [18]:
from Bio import motifs, SeqIO
from Bio.Seq import Seq
import numpy as np

In [19]:
sites = [
    "GAGGTAAAC",
    "TCCGTAAGC",
    "CAGGTTGGA",
    "ACAGTCAGC",
    "TAGGTCAGC",
    "CAGGTCAGC",
    "CAGGTCGAT",
    "CAGGTCAGC",
    "CAGGTCAGC",
    "CAGGTTGGC"
]

Создание мотива:

In [20]:
instances = [Seq(site) for site in sites]
m = motifs.create(instances)

print("Мотив создан:")
print("Консенсусная последовательность:", m.consensus)
print("Матрица частот:")
print(m.counts)

Мотив создан:
Консенсусная последовательность: CAGGTCAGC
Матрица частот:
        0      1      2      3      4      5      6      7      8
A:   1.00   8.00   1.00   0.00   0.00   2.00   7.00   2.00   1.00
C:   6.00   2.00   1.00   0.00   0.00   6.00   0.00   0.00   8.00
G:   1.00   0.00   8.00  10.00   0.00   0.00   3.00   8.00   0.00
T:   2.00   0.00   0.00   0.00  10.00   2.00   0.00   0.00   1.00



Используем геном асгардархеи (вместо первой хромосомы человека). Загрузка файла:

In [21]:
filename = "GCA_029856635.1_ASM2985663v1_genomic.fna"

with open(filename) as handle:
    for record in SeqIO.parse(handle, "fasta"):
        sequence = str(record.seq[:1000000]).upper()
        print(f"Загружено {len(sequence)} нуклеотидов из {record.id}")
        break  # Берем только первую запись

Загружено 37163 нуклеотидов из JAHKLH010000001.1


Поиск сайтов связывания с порогом > 5.0:

In [22]:
threshold = 5.0

def find_hits(sequence, motif, threshold, strand='+'):
    hits = []
    pssm = motif.pssm  # Получаем PSSM

    for position, score in pssm.search(sequence, threshold=threshold):
        hits.append({
            'position': position,
            'strand': strand,
            'score': round(score, 3)
        })
    return hits

Поиск на прямой цепи:

In [23]:
hits_forward = find_hits(sequence, m, threshold, '+')
print(f"Найдено хитов на прямой цепи: {len(hits_forward)}")

Найдено хитов на прямой цепи: 65


Поиск на обратной комплементарной цепи. Получаем обратную комплементарную последовательность. Корректируем позиции для обратной цепи. Позиция в оригинальной последовательности = длина - позиция_в_rev - длина_мотива

In [24]:
reverse_seq = str(Seq(sequence).reverse_complement())
hits_reverse = find_hits(reverse_seq, m, threshold, '-')

motif_length = len(m.consensus)
for hit in hits_reverse:
    hit['position'] = len(sequence) - hit['position'] - motif_length

print(f"Найдено хитов на обратной цепи: {len(hits_reverse)}")

Найдено хитов на обратной цепи: 65


Объединяем все хиты и сортируем по позиции:

In [25]:
all_hits = hits_forward + hits_reverse
all_hits.sort(key=lambda x: x['position'])

for hit in all_hits:
    print(f"{hit['position']:<10} {hit['strand']:<5} {hit['score']:<10}")

print(f"Всего найдено хитов: {len(all_hits)}")

-35918     +     5.915999889373779
-32696     +     5.138999938964844
-29298     +     6.138999938964844
-29206     +     6.330999851226807
-27669     +     5.553999900817871
-26128     +     5.553999900817871
-23299     +     6.138999938964844
-23251     +     8.138999938964844
-22186     +     5.553999900817871
-15744     +     5.915999889373779
-15136     +     5.553999900817871
-14965     +     5.553999900817871
-13482     +     5.553999900817871
-13393     +     6.330999851226807
-12264     +     5.553999900817871
-11987     +     5.915999889373779
-11750     +     5.553999900817871
-11695     +     5.553999900817871
-8895      +     5.553999900817871
-8352      +     7.553999900817871
-8299      +     9.138999938964844
-7964      +     5.553999900817871
-7314      +     7.138999938964844
-7097      +     5.553999900817871
-6780      +     8.916000366210938
-6585      +     5.138999938964844
-5672      +     6.330999851226807
-5081      +     6.915999889373779
-4971      +     10.